# Week 5 示例：内容安全场景的指标与阈值

- 作者：共享仓库示例
- 日期：2026-07-14
- 来源：`暑期居家集训学习计划.md` → Week 5 → 不良内容检测理论与方法
- 适用周次：Week 5
- 分类：NLP / 内容安全
- 关键词：Recall、Precision、F1、ROC-AUC、阈值
- 运行环境：Python 3.10+、NumPy、Matplotlib、scikit-learn
- 适合读者：已经理解分类任务和基本评估指标的初学者

## 学习目标

1. 观察类别不平衡场景下不同指标的变化。
2. 理解阈值对 Recall、Precision 和 F1 的影响。

> 原文 Week 5 以理论、综述阅读和 BERT 小规模实验要求为主，没有独立代码块；本例整理了原文强调的“Recall > Precision”和前文指标代码。

示例数据是模拟数据，不代表真实安全数据集的结果。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    precision_recall_fscore_support, roc_auc_score
)

rng = np.random.default_rng(42)
y_true = np.array([0] * 90 + [1] * 10)
scores = np.concatenate([
    rng.beta(2, 8, size=90),
    rng.beta(7, 2, size=10),
])
print('positive rate:', y_true.mean())
print('ROC-AUC:', round(roc_auc_score(y_true, scores), 4))

## 1. 改变阈值，观察 Recall 与 Precision 的权衡

安全场景通常更重视降低漏检，因此不能只看 Accuracy。阈值应该结合业务代价和验证集结果选择。

In [ ]:
thresholds = np.linspace(0.1, 0.9, 17)
rows = []
for threshold in thresholds:
    y_pred = (scores >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='binary', zero_division=0
    )
    rows.append((threshold, precision, recall, f1))

best_recall = max(rows, key=lambda row: (row[2], row[3]))
best_f1 = max(rows, key=lambda row: row[3])
print('Recall 优先的阈值、Precision、Recall、F1:', best_recall)
print('F1 最优的阈值、Precision、Recall、F1:', best_f1)

In [ ]:
rows = np.array(rows)
plt.figure(figsize=(7, 4))
plt.plot(rows[:, 0], rows[:, 1], label='Precision')
plt.plot(rows[:, 0], rows[:, 2], label='Recall')
plt.plot(rows[:, 0], rows[:, 3], label='F1')
plt.xlabel('Decision threshold')
plt.ylabel('Score')
plt.title('Metric trade-off in a content safety example')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 小结

完整 Week 5 还需要补充真实中文数据、BERT 微调、类别不平衡处理、对抗规避和视频理解相关内容。